# 90% compression, 2B tokens — CE loss: PCD vs HPD vs PCD with frozen P (no-P)

Plots `steps/100/train/loss` (100-step averaged **cross-entropy**, no distillation term) for:
- **PCD 90%** — Table 2(b), wandb `llm-random-test/k9ndr8i1` (LR 13)
- **HPD 90%** — Table 2(b), wandb `llm-random-test/pajazura` (LR 12)
- **PCD no-P 90%** — `freeze_projections: true`, HPD LR 12, wandb `pc_v2/0z8eaq4v`

Needs `WANDB_API_KEY` in the environment.

In [1]:
import pandas as pd
import plotly.express as px
import wandb

ENTITY = "ideas_cv"
METRIC = "steps/100/train/loss"

# label -> (project, run id). Order fixes legend order and colour.
RUNS = {
    "PCD 90% (LR 13)": ("llm-random-test", "k9ndr8i1"),
    "HPD 90% (LR 12)": ("llm-random-test", "pajazura"),
    "PCD no-P 90% (LR 12)": ("pc_v2", "0z8eaq4v"),
}
# first three slots of the validated categorical palette, fixed per series
COLORS = dict(zip(RUNS, ["#2a78d6", "#eb6834", "#1baf7a"]))

In [2]:
api = wandb.Api(timeout=120)

frames = []
for label, (project, run_id) in RUNS.items():
    run = api.run(f"{ENTITY}/{project}/{run_id}")
    rows = [
        {"step": r["_step"], "loss": r[METRIC]}
        for r in run.scan_history(keys=["_step", METRIC])
        if r.get(METRIC) is not None
    ]
    df = pd.DataFrame(rows)
    df["run"] = label
    frames.append(df)
    print(f"{label:22s} {run.state:9s} points={len(df):3d} last step={df.step.max()} last loss={df.loss.iloc[-1]:.4f}")

losses = pd.concat(frames, ignore_index=True)

PCD 90% (LR 13)        finished  points= 40 last step=4001 last loss=3.1312


HPD 90% (LR 12)        finished  points= 40 last step=4001 last loss=3.2448


PCD no-P 90% (LR 12)   finished  points= 40 last step=4001 last loss=3.2214


In [3]:
fig = px.line(
    losses, x="step", y="loss", color="run",
    color_discrete_map=COLORS, category_orders={"run": list(RUNS)},
    markers=True, template="plotly_white",
    labels={"step": "Training step", "loss": "Cross-entropy loss (100-step avg)", "run": ""},
    title="Llama 3 1B, 90% compression, 2B tokens — training CE loss",
)
fig.update_traces(line_width=2, marker_size=8)
fig.update_layout(hovermode="x unified", legend=dict(orientation="h", y=1.02, yanchor="bottom", x=0))
fig.show()

In [4]:
# Same curves, second half only (loss differences are small there)
late = losses[losses.step >= 2000]
fig2 = px.line(
    late, x="step", y="loss", color="run",
    color_discrete_map=COLORS, category_orders={"run": list(RUNS)},
    markers=True, template="plotly_white",
    labels={"step": "Training step", "loss": "Cross-entropy loss (100-step avg)", "run": ""},
    title="Steps 2000+ (zoom)",
)
fig2.update_traces(line_width=2, marker_size=8)
fig2.update_layout(hovermode="x unified", legend=dict(orientation="h", y=1.02, yanchor="bottom", x=0))
fig2.show()

In [5]:
# Final / latest loss per run (table view)
losses.sort_values("step").groupby("run", sort=False).tail(1).set_index("run").loc[list(RUNS)]

,step,loss
run,,
PCD 90% (LR 13),4001,3.131167
HPD 90% (LR 12),4001,3.244824
PCD no-P 90% (LR 12),4001,3.221406
